# Tutorial · Controller SDN dengan **Ryu** di atas Mininet

Tutorial ini fokus pada **Ryu sebagai controller OpenFlow 1.3**. Berbeda dengan tutorial Mininet (switch tanpa controller), di sini logika jaringan ditentukan oleh **aplikasi Ryu** yang kita tulis dan jalankan dengan `ryu-manager`, sedangkan Mininet hanya menyediakan switch & host.

Yang dibangun bertahap:
1. **`l2_switch.py`** — learning switch OpenFlow 1.3 (dasar PacketIn/FlowMod).
2. **`topo_router.py`** — controller yang **menemukan topologi** (LLDP) lalu memasang **rute shortest-path** dengan NetworkX; aman untuk topologi ber-loop tanpa STP.
3. **Monitoring** — `topo_router.py` mengumpulkan **flow stats** & **port stats** tiap interval ke CSV, lalu kita ekspor sebagai dataset.
4. **REST & mitigasi** — endpoint untuk melihat topologi dan **memasang drop rule** (blokir port) sebagai dasar mitigasi DDoS.

Keluaran yang dapat dipakai lanjutan:
- **optimasi** → topologi yang ditemukan controller diekspor ke JSON (format `data/topology.json`) untuk `01_optimasi_jaringan.ipynb`;
- **deteksi DDoS** → `flow_stats.csv`/`port_stats.csv` dari controller, diproses dengan pipeline `02_deteksi_ddos.ipynb`.

> Jalankan Jupyter sebagai root (`bash run_jupyter.sh` / `run_jupyter.bat wsl`).

In [1]:
# ============================ KONFIGURASI ============================
OF_PORT = 6653
REST_PORT = 8080
STATS_INTERVAL = 2.0

import os, sys, json, time, re, shutil, platform, subprocess, itertools, signal
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE / "results" / "tutorial_ryu"
RUNTIME = BASE / "runtime" / "tutorial_ryu"
RYU_DIR = BASE / "ryu_apps"
DATA = BASE / "data"
for d in (OUT, RUNTIME, RYU_DIR, DATA):
    d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

if platform.system() != "Linux" or os.geteuid() != 0:
    raise SystemExit("Butuh Linux/WSL2 + Jupyter sebagai root (bash run_jupyter.sh).")
from mininet.net import Mininet
from mininet.node import RemoteController, OVSSwitch
from mininet.link import TCLink
from mininet.topo import Topo
from mininet.clean import cleanup
from mininet.log import setLogLevel
setLogLevel("warning")

def ryu_manager():
    c = Path(sys.executable).parent / "ryu-manager"
    return str(c) if c.exists() else shutil.which("ryu-manager")
OVS_DATAPATH = "kernel" if (os.path.isdir("/sys/module/openvswitch") or
    (shutil.which("modprobe") and subprocess.run(["modprobe", "openvswitch"], capture_output=True).returncode == 0)) else "user"
assert ryu_manager(), "ryu-manager tidak ditemukan di venv"
print("ryu-manager:", ryu_manager(), "| datapath OVS:", OVS_DATAPATH)

ryu-manager: /home/wirar/.venvs/sdn-ryu-mininet/bin/ryu-manager | datapath OVS: kernel


## 1. Aplikasi dasar: learning switch `l2_switch.py`
Prinsip controller OpenFlow:
- switch yang belum tahu tujuan mengirim paket ke controller (**PacketIn**);
- controller mempelajari `MAC sumber → port`, lalu bila tujuan diketahui memasang **FlowMod** agar paket berikutnya ditangani switch tanpa lewat controller;
- bila tujuan belum diketahui, paket di-*flood*.

Ini fondasi semua aplikasi berikutnya.

In [2]:
L2 = r'''
"""l2_switch.py - learning switch OpenFlow 1.3 (contoh dasar controller Ryu)."""
from ryu.base import app_manager
from ryu.controller import ofp_event
from ryu.controller.handler import CONFIG_DISPATCHER, MAIN_DISPATCHER, set_ev_cls
from ryu.ofproto import ofproto_v1_3
from ryu.lib.packet import packet, ethernet, ether_types


class L2Switch(app_manager.RyuApp):
    OFP_VERSIONS = [ofproto_v1_3.OFP_VERSION]

    def __init__(self, *args, **kwargs):
        super(L2Switch, self).__init__(*args, **kwargs)
        self.mac_to_port = {}

    def add_flow(self, dp, priority, match, actions, idle=0):
        ofp, parser = dp.ofproto, dp.ofproto_parser
        inst = [parser.OFPInstructionActions(ofp.OFPIT_APPLY_ACTIONS, actions)]
        dp.send_msg(parser.OFPFlowMod(datapath=dp, priority=priority, match=match,
                                      instructions=inst, idle_timeout=idle))

    @set_ev_cls(ofp_event.EventOFPSwitchFeatures, CONFIG_DISPATCHER)
    def features(self, ev):
        dp = ev.msg.datapath
        parser, ofp = dp.ofproto_parser, dp.ofproto
        # table-miss: kirim ke controller
        self.add_flow(dp, 0, parser.OFPMatch(),
                      [parser.OFPActionOutput(ofp.OFPP_CONTROLLER, ofp.OFPCML_NO_BUFFER)])

    @set_ev_cls(ofp_event.EventOFPPacketIn, MAIN_DISPATCHER)
    def packet_in(self, ev):
        msg = ev.msg
        dp = msg.datapath
        ofp, parser = dp.ofproto, dp.ofproto_parser
        in_port = msg.match["in_port"]
        eth = packet.Packet(msg.data).get_protocol(ethernet.ethernet)
        if eth is None or eth.ethertype == ether_types.ETH_TYPE_LLDP:
            return
        self.mac_to_port.setdefault(dp.id, {})[eth.src] = in_port
        out_port = self.mac_to_port[dp.id].get(eth.dst, ofp.OFPP_FLOOD)
        actions = [parser.OFPActionOutput(out_port)]
        if out_port != ofp.OFPP_FLOOD:
            self.add_flow(dp, 1, parser.OFPMatch(eth_dst=eth.dst, eth_src=eth.src), actions, idle=30)
        data = msg.data if msg.buffer_id == ofp.OFP_NO_BUFFER else None
        dp.send_msg(parser.OFPPacketOut(datapath=dp, buffer_id=msg.buffer_id,
                                        in_port=in_port, actions=actions, data=data))
'''
(RYU_DIR / "l2_switch.py").write_text(L2.lstrip(), encoding="utf-8")
print("ditulis:", RYU_DIR / "l2_switch.py")

ditulis: /mnt/c/Users/wirar/OneDrive/Documents/GitHub/sdn/ryu_apps/l2_switch.py


## 2. Menjalankan controller + Mininet dari notebook
Pola yang dipakai seluruh tutorial: controller dijalankan sebagai proses `ryu-manager` terpisah yang mendengarkan port 6653; Mininet menghubungkan switch ke `RemoteController`. Helper di bawah menangani start/stop, pembersihan, dan (pada datapath userspace) mematikan TX checksum offload agar TCP tidak dibuang.

In [3]:
class Scenario:
    """Kelola satu sesi: proses ryu-manager + jaringan Mininet dari graf NetworkX."""
    def __init__(self, G, apps, extra_args=None):
        self.G, self.apps, self.extra = G, apps, extra_args or []
        self.net = self.ryu = None

    def start(self, wait=4):
        self.stop()
        subprocess.run(["fuser", "-k", f"{OF_PORT}/tcp", f"{REST_PORT}/tcp"], capture_output=True)
        env = dict(os.environ, SDN_OUT=str(RUNTIME), SDN_STATS_INTERVAL=str(STATS_INTERVAL))
        cmd = [ryu_manager(), "--ofp-tcp-listen-port", str(OF_PORT),
               "--wsapi-port", str(REST_PORT), *self.extra, *[str(RYU_DIR / a) for a in self.apps]]
        self.ryu = subprocess.Popen(cmd, stdout=open(RUNTIME / "ryu.log", "w"),
                                    stderr=subprocess.STDOUT, env=env)
        time.sleep(wait)
        if self.ryu.poll() is not None:
            raise RuntimeError("ryu-manager berhenti; lihat runtime/tutorial_ryu/ryu.log")
        self.net = build_net(self.G)
        return self

    def stop(self):
        if self.net:
            try:
                self.net.stop()
            except Exception:
                pass
            self.net = None
        if self.ryu:
            self.ryu.send_signal(signal.SIGINT)
            try:
                self.ryu.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.ryu.kill()
            self.ryu = None
        cleanup()

def build_net(G):
    net = Mininet(switch=partial(OVSSwitch, protocols="OpenFlow13", datapath=OVS_DATAPATH),
                  link=TCLink, controller=None, autoSetMacs=False, build=False)
    net.addController("c0", controller=RemoteController, ip="127.0.0.1", port=OF_PORT)
    sw = [n for n in G if G.nodes[n]["type"] == "switch"]
    for s in sorted(sw, key=lambda s: int(s[1:])):
        net.addSwitch(s, dpid=f"{int(s[1:]):016x}")
    for h in [n for n in G if G.nodes[n]["type"] == "host"]:
        net.addHost(h, ip=G.nodes[h]["ip"] + "/24", mac=G.nodes[h]["mac"])
    for u, v, d in G.edges(data=True):
        opts = dict(bw=d.get("bandwidth", 100))
        if d.get("delay", 0) > 0:
            opts["delay"] = f"{d['delay']}ms"
        if d.get("loss", 0) > 0:
            opts["loss"] = d["loss"] * 100
        net.addLink(u, v, **opts)
    net.build(); net.start()
    if OVS_DATAPATH == "user":
        for h in net.hosts:
            for intf in h.intfList():
                h.cmd(f"ethtool -K {intf} tx off > /dev/null 2>&1")
    return net

def simple_topo(kind="tree", **kw):
    """Graf switch+host sederhana (dipakai jika tutorial Mininet tidak diimpor)."""
    G = nx.Graph(kind=kind)
    if kind == "tree":
        depth, fanout = kw.get("depth", 2), kw.get("fanout", 2)
        T = nx.relabel_nodes(nx.balanced_tree(fanout, depth), lambda u: f"s{u+1}")
        leaves = [u for u in T if T.degree(u) == 1]
    elif kind == "ring":
        n = kw.get("n", 4); T = nx.Graph(); nx.add_cycle(T, [f"s{i+1}" for i in range(n)]); leaves = list(T)
    else:
        n = kw.get("n", 4); T = nx.relabel_nodes(nx.complete_graph(n), lambda u: f"s{u+1}"); leaves = list(T)
    for u in T:
        G.add_node(u, type="switch")
    for u, v in T.edges():
        G.add_edge(u, v, delay=2.0, bandwidth=50.0, loss=0.0)
    h = 0
    for sw in sorted(leaves, key=lambda s: int(s[1:])):
        h += 1
        G.add_node(f"h{h}", type="host", ip=f"10.0.0.{h}", mac=f"00:00:00:00:00:{h:02x}")
        G.add_edge(f"h{h}", sw, delay=0.0, bandwidth=1000.0, loss=0.0)
    return G

# uji learning switch pada topologi tree
G_tree = simple_topo("tree", depth=2, fanout=2)
sc = Scenario(G_tree, ["l2_switch.py"]).start()
loss = sc.net.pingAll(timeout=2)
print(f"l2_switch.py — pingAll loss {loss:.0f}% (0% = controller berfungsi)")
print(sc.net.get("s1").cmd("ovs-ofctl -O OpenFlow13 dump-flows s1 | head -5"))
sc.stop()

*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb:

l2_switch.py — pingAll loss 0% (0% = controller berfungsi)
OFPST_FLOW reply (OF1.3) (xid=0x2):
 cookie=0x0, duration=0.304s, table=0, n_packets=2, n_bytes=196, idle_timeout=30, priority=1,dl_src=00:00:00:00:00:03,dl_dst=00:00:00:00:00:01 actions=output:1
 cookie=0x0, duration=0.291s, table=0, n_packets=1, n_bytes=98, idle_timeout=30, priority=1,dl_src=00:00:00:00:00:01,dl_dst=00:00:00:00:00:03 actions=output:2
 cookie=0x0, duration=0.254s, table=0, n_packets=1, n_bytes=98, idle_timeout=30, priority=1,dl_src=00:00:00:00:00:04,dl_dst=00:00:00:00:00:01 actions=output:1
 cookie=0x0, duration=0.242s, table=0, n_packets=0, n_bytes=0, idle_timeout=30, priority=1,dl_src=00:00:00:00:00:01,dl_dst=00:00:00:00:00:04 actions=output:2



## 3. Controller rute shortest-path + penemuan topologi: `topo_router.py`
Aplikasi ini:
- memakai modul **`ryu.topology`** untuk mengetahui switch & link (LLDP) — dijalankan dengan `--observe-links`;
- membangun graf dengan **NetworkX**, menghitung **shortest-path** antar switch, dan memasang rute per `eth_dst` host → **aman untuk topologi ber-loop tanpa STP** (paket hanya mengikuti pohon rute);
- mengumpulkan **flow stats** & **port stats** tiap interval ke CSV;
- menyediakan **REST**: `GET /sdn/topology`, `GET /sdn/stats`, `POST /sdn/block`.

Host dipelajari dari PacketIn ARP; untuk topologi ber-loop, paket ARP yang belum dikenal dibatasi agar tidak membanjiri (dikirim hanya ke port tepi).

In [4]:
ROUTER = r'''
"""topo_router.py - Ryu OF1.3: penemuan topologi (LLDP) + rute shortest-path (NetworkX)
+ monitoring flow/port stats + REST (topologi, stats, blokir port)."""
import json
import os
import time

import networkx as nx
from ryu.base import app_manager
from ryu.controller import ofp_event
from ryu.controller.handler import CONFIG_DISPATCHER, MAIN_DISPATCHER, DEAD_DISPATCHER, set_ev_cls
from ryu.lib import hub
from ryu.lib.packet import packet, ethernet, ether_types, arp, ipv4
from ryu.ofproto import ofproto_v1_3
from ryu.topology import event as topo_event
from ryu.topology.api import get_switch, get_link
from ryu.app.wsgi import ControllerBase, WSGIApplication, route
from webob import Response

OUT = os.environ.get("SDN_OUT", "/tmp")
INTERVAL = float(os.environ.get("SDN_STATS_INTERVAL", "2.0"))
INSTANCE = "topo_router_app"


class TopoRouter(app_manager.RyuApp):
    OFP_VERSIONS = [ofproto_v1_3.OFP_VERSION]
    _CONTEXTS = {"wsgi": WSGIApplication}

    def __init__(self, *args, **kwargs):
        super(TopoRouter, self).__init__(*args, **kwargs)
        self.dps = {}
        self.net = nx.DiGraph()          # switch-switch (port) + host
        self.host_port = {}              # mac -> (dpid, port)
        self.host_ip = {}                # ip -> mac
        self.flow_rows, self.port_rows = [], []
        self.req_ts = {}
        kwargs["wsgi"].register(RestApi, {INSTANCE: self})
        hub.spawn(self._monitor)

    def add_flow(self, dp, priority, match, actions, idle=0, hard=0, cookie=0):
        ofp, parser = dp.ofproto, dp.ofproto_parser
        inst = [parser.OFPInstructionActions(ofp.OFPIT_APPLY_ACTIONS, actions)] if actions else []
        dp.send_msg(parser.OFPFlowMod(datapath=dp, priority=priority, match=match,
                                      instructions=inst, idle_timeout=idle, hard_timeout=hard, cookie=cookie))

    @set_ev_cls(ofp_event.EventOFPSwitchFeatures, CONFIG_DISPATCHER)
    def features(self, ev):
        dp = ev.msg.datapath
        self.dps[dp.id] = dp
        parser, ofp = dp.ofproto_parser, dp.ofproto
        self.add_flow(dp, 0, parser.OFPMatch(),
                      [parser.OFPActionOutput(ofp.OFPP_CONTROLLER, ofp.OFPCML_NO_BUFFER)])

    @set_ev_cls(topo_event.EventSwitchEnter)
    def _topology(self, ev):
        hub.sleep(0.3)
        switches = [s.dp.id for s in get_switch(self)]
        self.net = nx.DiGraph()
        self.net.add_nodes_from(switches)
        for l in get_link(self):
            self.net.add_edge(l.src.dpid, l.dst.dpid, port=l.src.port_no)
        self.logger.info("topologi: %d switch, %d link", len(switches), self.net.number_of_edges())

    def _route(self, dp, in_port, pkt_eth, ip_src, ip_dst):
        """Pasang rute shortest-path dari switch sekarang menuju switch tempat host tujuan berada."""
        dst_mac = pkt_eth.dst
        if dst_mac not in self.host_port:
            return None
        dst_dpid, dst_port = self.host_port[dst_mac]
        if dp.id == dst_dpid:
            out = dst_port
        else:
            if not self.net.has_node(dp.id) or not self.net.has_node(dst_dpid) \
               or not nx.has_path(self.net, dp.id, dst_dpid):
                return None
            path = nx.shortest_path(self.net, dp.id, dst_dpid)
            out = self.net[dp.id][path[1]]["port"]
        parser = dp.ofproto_parser
        match = parser.OFPMatch(eth_type=ether_types.ETH_TYPE_IP, ipv4_src=ip_src, ipv4_dst=ip_dst)
        self.add_flow(dp, 10, match, [parser.OFPActionOutput(out)], idle=30)
        return out

    @set_ev_cls(ofp_event.EventOFPPacketIn, MAIN_DISPATCHER)
    def packet_in(self, ev):
        msg = ev.msg
        dp = msg.datapath
        ofp, parser = dp.ofproto, dp.ofproto_parser
        in_port = msg.match["in_port"]
        pkt = packet.Packet(msg.data)
        eth = pkt.get_protocol(ethernet.ethernet)
        if eth is None or eth.ethertype in (ether_types.ETH_TYPE_LLDP, ether_types.ETH_TYPE_IPV6):
            return
        # pelajari host dari port tepi (port yang bukan antar-switch)
        switch_ports = {d["port"] for _, _, d in self.net.out_edges(dp.id, data=True)} if self.net.has_node(dp.id) else set()
        if in_port not in switch_ports:
            self.host_port[eth.src] = (dp.id, in_port)
            a = pkt.get_protocol(arp.arp)
            if a:
                self.host_ip[a.src_ip] = eth.src
        out_port = None
        ip4 = pkt.get_protocol(ipv4.ipv4)
        if ip4 is not None:
            out_port = self._route(dp, in_port, eth, ip4.src, ip4.dst)
        if out_port is None:
            # ARP / tujuan belum dikenal: flood hanya ke port tepi (hindari badai pada loop)
            out_port = ofp.OFPP_FLOOD
        actions = [parser.OFPActionOutput(out_port)]
        data = msg.data if msg.buffer_id == ofp.OFP_NO_BUFFER else None
        dp.send_msg(parser.OFPPacketOut(datapath=dp, buffer_id=msg.buffer_id,
                                        in_port=in_port, actions=actions, data=data))

    # ---------------- monitoring ----------------
    def _monitor(self):
        while True:
            now = time.time()
            for dp in list(self.dps.values()):
                self.req_ts[dp.id] = now
                dp.send_msg(dp.ofproto_parser.OFPFlowStatsRequest(dp, table_id=0))
                dp.send_msg(dp.ofproto_parser.OFPPortStatsRequest(dp, 0, dp.ofproto.OFPP_ANY))
            self._flush()
            hub.sleep(INTERVAL)

    def _flush(self):
        if self.flow_rows:
            with open(os.path.join(OUT, "flow_stats.csv"), "a") as f:
                for r in self.flow_rows:
                    f.write(",".join(map(str, r)) + "\n")
            self.flow_rows = []
        if self.port_rows:
            with open(os.path.join(OUT, "port_stats.csv"), "a") as f:
                for r in self.port_rows:
                    f.write(",".join(map(str, r)) + "\n")
            self.port_rows = []

    @set_ev_cls(ofp_event.EventOFPFlowStatsReply, MAIN_DISPATCHER)
    def _flow_reply(self, ev):
        ts = round(self.req_ts.get(ev.msg.datapath.id, time.time()), 3)
        for st in ev.msg.body:
            if st.priority != 10:
                continue
            m = st.match
            self.flow_rows.append([ts, ev.msg.datapath.id, m.get("ipv4_src"), m.get("ipv4_dst"),
                                   st.packet_count, st.byte_count, round(st.duration_sec + st.duration_nsec / 1e9, 3)])

    @set_ev_cls(ofp_event.EventOFPPortStatsReply, MAIN_DISPATCHER)
    def _port_reply(self, ev):
        ts = round(self.req_ts.get(ev.msg.datapath.id, time.time()), 3)
        for st in ev.msg.body:
            if st.port_no <= ev.msg.datapath.ofproto.OFPP_MAX:
                self.port_rows.append([ts, ev.msg.datapath.id, st.port_no, st.rx_packets, st.tx_packets,
                                       st.rx_bytes, st.tx_bytes, st.rx_dropped, st.tx_dropped])

    @set_ev_cls(ofp_event.EventOFPStateChange, [MAIN_DISPATCHER, DEAD_DISPATCHER])
    def _state(self, ev):
        if ev.state == DEAD_DISPATCHER and ev.datapath.id in self.dps:
            del self.dps[ev.datapath.id]

    def block_port(self, dpid, in_port, seconds=0):
        dp = self.dps.get(int(dpid))
        if not dp:
            return False
        parser = dp.ofproto_parser
        self.add_flow(dp, 1000, parser.OFPMatch(in_port=int(in_port)), [], hard=int(seconds), cookie=0xDEAD)
        return True

    def topo_dict(self):
        return {"switches": sorted(self.net.nodes()),
                "links": [{"src": u, "dst": v, "port": d["port"]} for u, v, d in self.net.edges(data=True)],
                "hosts": {ip: {"mac": mac, "dpid": self.host_port.get(mac, (None, None))[0],
                               "port": self.host_port.get(mac, (None, None))[1]}
                          for ip, mac in self.host_ip.items()}}


class RestApi(ControllerBase):
    def __init__(self, req, link, data, **config):
        super(RestApi, self).__init__(req, link, data, **config)
        self.app = data[INSTANCE]

    @route("sdn", "/sdn/topology", methods=["GET"])
    def topology(self, req, **kw):
        return Response(content_type="application/json", body=json.dumps(self.app.topo_dict()))

    @route("sdn", "/sdn/block", methods=["POST"])
    def block(self, req, **kw):
        body = json.loads(req.body or b"{}")
        ok = self.app.block_port(body["dpid"], body["in_port"], body.get("seconds", 0))
        return Response(content_type="application/json", body=json.dumps({"blocked": ok}))
'''
(RYU_DIR / "topo_router.py").write_text(ROUTER.lstrip(), encoding="utf-8")
# siapkan header CSV
(RUNTIME / "flow_stats.csv").write_text("ts,dpid,ipv4_src,ipv4_dst,packet_count,byte_count,duration\n")
(RUNTIME / "port_stats.csv").write_text("ts,dpid,port,rx_packets,tx_packets,rx_bytes,tx_bytes,rx_dropped,tx_dropped\n")
print("ditulis:", RYU_DIR / "topo_router.py")

ditulis: /mnt/c/Users/wirar/OneDrive/Documents/GitHub/sdn/ryu_apps/topo_router.py


## 4. Menjalankan `topo_router.py` di topologi ber-loop
Topologi **ring** sengaja dipilih karena punya loop: tanpa controller yang benar, ARP akan membanjir tanpa henti. `topo_router.py` menemukan topologi via LLDP (`--observe-links`) lalu merutekan paket mengikuti shortest-path, sehingga loop aman tanpa STP.

In [5]:
G_ring = simple_topo("ring", n=4)
sc = Scenario(G_ring, ["topo_router.py"], extra_args=["--observe-links"]).start(wait=6)
time.sleep(3)                                   # beri waktu LLDP menemukan semua link
print("pingAll loss:", sc.net.pingAll(timeout=3), "%")
time.sleep(STATS_INTERVAL * 2)

# ambil topologi hasil penemuan controller lewat REST
import urllib.request
topo = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{REST_PORT}/sdn/topology", timeout=5).read())
print(json.dumps(topo, indent=1)[:600])
sc.stop()

*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb: quantum of class 50001 is big. Consider r2q change.
*** Error: Warning: sch_htb:

pingAll loss: 0.0 %


HTTPError: HTTP Error 500: Internal Server Error

## 5. Ekspor topologi yang ditemukan → dataset optimasi
Topologi hasil penemuan controller (switch, link, host) diubah ke JSON berformat `data/topology.json`. Karena controller tidak mengetahui delay/bandwidth link secara langsung, atribut diisi dari **cetak biru** yang kita pakai membangun Mininet (di dunia nyata, nilai ini berasal dari konfigurasi atau statistik port).

In [6]:
def export_discovered(G_blueprint, topo, path, description):
    sw = sorted({f"s{d}" for d in topo["switches"]}, key=lambda s: int(s[1:]))
    edges = []
    seen = set()
    for l in topo["links"]:
        u, v = f"s{l['src']}", f"s{l['dst']}"
        key = tuple(sorted((u, v)))
        if key in seen:
            continue
        seen.add(key)
        d = G_blueprint.get_edge_data(u, v) or {}
        edges.append(dict(source=u, target=v, delay=d.get("delay", 2.0),
                          bandwidth=d.get("bandwidth", 50.0), loss=d.get("loss", 0.0)))
    data = {"description": description, "nodes": sw, "edges": edges}
    Path(path).write_text(json.dumps(data, indent=1))
    return data

topo_json = export_discovered(G_ring, topo, DATA / "topologi_ring_ryu.json",
                              "Topologi ring hasil penemuan controller Ryu (atribut link dari cetak biru)")
print("ekspor:", DATA / "topologi_ring_ryu.json", "-", len(topo_json["edges"]), "link")
G_disc = nx.Graph()
for e in topo_json["edges"]:
    G_disc.add_edge(e["source"], e["target"])
fig, ax = plt.subplots(figsize=(4.5, 4))
nx.draw(G_disc, nx.circular_layout(G_disc), ax=ax, with_labels=True, node_color="#2f6db5",
        font_color="white", node_size=600)
ax.set_title("Topologi yang ditemukan controller")
plt.tight_layout(); plt.savefig(OUT / "topologi_ditemukan.png", dpi=150); plt.show()

NameError: name 'topo' is not defined

## 6. Monitoring → dataset untuk optimasi & deteksi DDoS
Controller sudah menulis `flow_stats.csv` dan `port_stats.csv` selama sesi berjalan. Sel berikut menjalankan sesi singkat dengan trafik `iperf` lalu menghitung **utilisasi link** dari port stats (berguna sebagai bobot biaya pada algoritma optimasi), dan menyiapkan `flow_stats.csv` sebagai masukan pipeline deteksi DDoS.

In [ ]:
for f in ("flow_stats.csv", "port_stats.csv"):
    (RUNTIME / f).write_text(open(RUNTIME / f).readline())     # sisakan header saja

sc = Scenario(simple_topo("tree", depth=2, fanout=2), ["topo_router.py"], extra_args=["--observe-links"]).start(wait=6)
time.sleep(2); sc.net.pingAll(timeout=2)
h = sc.net.hosts
t0 = time.time()
h[-1].cmd("iperf -s -p 5001 > /dev/null 2>&1 &"); time.sleep(0.5)
h[0].cmd(f"timeout 18 iperf -c {h[-1].IP()} -p 5001 -t 12 -f m > /dev/null 2>&1 &")
time.sleep(14)
t1 = time.time()
sc.stop()

port = pd.read_csv(RUNTIME / "port_stats.csv")
port = port[(port.ts >= t0) & (port.ts <= t1)]
util = []
for (dpid, p), g in port.groupby(["dpid", "port"]):
    g = g.sort_values("ts")
    if len(g) >= 2:
        dt = g.ts.iloc[-1] - g.ts.iloc[0]
        mbps = (g.tx_bytes.iloc[-1] - g.tx_bytes.iloc[0]) * 8 / dt / 1e6 if dt > 0 else 0
        util.append(dict(dpid=dpid, port=p, tx_mbps=round(mbps, 2)))
util = pd.DataFrame(util).sort_values("tx_mbps", ascending=False)
print("Utilisasi port tertinggi (dari port stats controller):")
display(util.head(8))
flow = pd.read_csv(RUNTIME / "flow_stats.csv")
print(f"\nflow_stats.csv: {len(flow)} baris — masukan untuk pipeline deteksi di 02_deteksi_ddos.ipynb")
display(flow.tail(5))

## 7. Mitigasi: blokir port lewat REST
Endpoint `POST /sdn/block` memasang drop rule prioritas tinggi pada `(dpid, in_port)` — dasar mitigasi DDoS: begitu pipeline deteksi (`02_deteksi_ddos.ipynb`) menandai port sumber serangan, controller dapat memblokirnya. Di sini kita tunjukkan mekanismenya: blokir port sebuah host, lalu buktikan host itu kehilangan konektivitas sementara host lain tetap terhubung.

In [ ]:
import urllib.request
sc = Scenario(simple_topo("tree", depth=2, fanout=2), ["topo_router.py"], extra_args=["--observe-links"]).start(wait=6)
time.sleep(2); sc.net.pingAll(timeout=2)
h1, h2, h3 = sc.net.hosts[0], sc.net.hosts[1], sc.net.hosts[-1]
# cari dpid & port tepi untuk h1 dari topologi controller
topo = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{REST_PORT}/sdn/topology", timeout=5).read())
info = topo["hosts"].get(h1.IP(), {})
print("Sebelum blokir:  h1→h3 loss", sc.net.ping([h1, h3], timeout=2), "%")
body = json.dumps({"dpid": info.get("dpid"), "in_port": info.get("port"), "seconds": 30}).encode()
resp = urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{REST_PORT}/sdn/block", data=body,
                              headers={"Content-Type": "application/json"}), timeout=5).read()
print("POST /sdn/block →", resp.decode())
time.sleep(1)
print("Setelah blokir:  h1→h3 loss", sc.net.ping([h1, h3], timeout=2), "% (h1 terputus)")
print("Host lain   :  h2→h3 loss", sc.net.ping([h2, h3], timeout=2), "% (tetap terhubung)")
sc.stop()

## 8. Penutup
File hasil di `results/tutorial_ryu/`, CSV monitoring di `runtime/tutorial_ryu/`, aplikasi Ryu di `ryu_apps/`, topologi di `data/`.

**Rangkuman aplikasi Ryu yang dibuat:**
- `l2_switch.py` — learning switch (dasar PacketIn/FlowMod).
- `topo_router.py` — penemuan topologi + rute shortest-path (NetworkX) + monitoring flow/port stats + REST (topologi, blokir port).

**Jalur ke notebook lain:**
- optimasi → `data/topologi_*_ryu.json` dan utilisasi link dari port stats;
- deteksi DDoS → `runtime/tutorial_ryu/flow_stats.csv` + pipeline fitur/model `02_deteksi_ddos.ipynb`.

In [ ]:
cleanup()
print("Aplikasi Ryu:")
for f in sorted(RYU_DIR.glob("*.py")):
    print("  ryu_apps/" + f.name)
print("\nHasil:")
for f in sorted(OUT.iterdir()):
    print("  " + str(f.relative_to(BASE)))

### Ide eksplorasi
1. Jalankan `ryu.app.gui_topology.gui_topology` dengan `--observe-links` lalu buka `http://127.0.0.1:8080` untuk melihat topologi secara visual.
2. Tambah bobot delay pada `_route` (`nx.shortest_path(..., weight=...)`) memakai delay dari konfigurasi link, sehingga rute mengikuti algoritma optimasi `01_optimasi_jaringan.ipynb`.
3. Gabungkan dengan `ryu.app.ofctl_rest` untuk melihat/menghapus flow lewat REST standar Ryu.
4. Pada `topo_router.py`, ubah granularitas flow menjadi per `(ipv4_src, ipv4_dst, ip_proto)` agar lebih dekat ke fitur deteksi DDoS.